# XY model

This notebook is part of the computational resources for the Statistical Physics course at École Polytechnique. To return to the main repository, follow this link: [https://github.com/cossio/StatPhysCompX](https://github.com/cossio/StatPhysCompX).

We consider the XY model defined on a 2-dimensional grid lattice. Each site is occupied by a circular spin, represented by an angle $\theta_i \in [0, 2\pi]$. The model is then defined by the Hamiltonian:

$$H = -J\sum_{(ij)} \cos(\theta_i - \theta_j)$$

where the sum traverses pairs of adjacent spins $(ij)$ in the grid.

The model exhibits a phase transition, where for high values of $J$ (equivalently, low temperatures), paired couples of *vortices* appear. If one attaches an arrow to every spin pointing in the direction indicated by the angle $\theta_i$, then a vortex is a point in the plane such that neighboring spins are "spiraling" towards it (or away from it, for an anti-vortex). If one draws a small circle around a vortex, the angles $\theta_i$ of the spins on the circumference traverse the full range, from $0$ to $2\pi$.

We will simulate this model using the Metropolis algorithm (like for the Ising model).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from numba import njit

In [ ]:
@njit
def _xy_energy(spins, L):
    """Compute the total energy of the XY lattice."""
    E = 0.0
    for i in range(L):
        for j in range(L):
            theta = spins[i, j]
            E -= np.cos(theta - spins[i, (j+1)%L]) + np.cos(theta - spins[(i+1)%L, j])
    return E


@njit
def _xy_metropolis_sweep(spins, J, L, n_steps):
    """Perform n_steps Metropolis updates on the XY model (in-place)."""
    for _ in range(n_steps):
        i = np.random.randint(0, L)
        j = np.random.randint(0, L)
        theta_new = 2.0 * np.pi * np.random.random()
        theta_old = spins[i, j]

        nb1 = spins[i, (j+1)%L]
        nb2 = spins[i, (j-1)%L]
        nb3 = spins[(i+1)%L, j]
        nb4 = spins[(i-1)%L, j]

        dE = 0.0
        dE -= np.cos(theta_new - nb1) - np.cos(theta_old - nb1)
        dE -= np.cos(theta_new - nb2) - np.cos(theta_old - nb2)
        dE -= np.cos(theta_new - nb3) - np.cos(theta_old - nb3)
        dE -= np.cos(theta_new - nb4) - np.cos(theta_old - nb4)
        dE *= J

        if dE <= 0.0 or np.random.random() < np.exp(-dE):
            spins[i, j] = theta_new


def xy_metropolis(J, L, steps_between_frames, number_of_frames):
    """Run XY Metropolis simulation. Returns (final_config, energy_per_spin_trace)."""
    spins = 2.0 * np.pi * np.random.rand(L, L)
    N = L * L
    energy_trace = np.zeros(number_of_frames)
    energy_trace[0] = _xy_energy(spins, L) / N
    for f in range(1, number_of_frames):
        _xy_metropolis_sweep(spins, J, L, steps_between_frames)
        energy_trace[f] = _xy_energy(spins, L) / N
    return spins.copy(), energy_trace

In [ ]:
# Warm up numba JIT
_ = xy_metropolis(1.0, 8, 100, 10)

In [ ]:
values_of_J = [0.5, 1, 3, 4]

simulations = {}
for J in values_of_J:
    np.random.seed(1)
    print(f"Simulating J={J}...")
    snap, energy = xy_metropolis(J, L=200, steps_between_frames=2000, number_of_frames=5000)
    simulations[J] = {"snapshot": snap, "energy": energy}
    print(f"  done.")

To check that our simulations have equilibrated, we can inspect the evolution of the energy per spin in time.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 8))
for ax, J in zip(axes, values_of_J):
    ax.plot(simulations[J]["energy"], linewidth=0.5)
    ax.set_title(f"J = {J}")
    ax.set_xlabel("time")
    ax.set_ylabel("energy per spin")
plt.tight_layout()
plt.show()

In [ ]:
# Show typical snapshots using HSV colormap (angle -> hue)
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, J in zip(axes.flat, values_of_J):
    spins = simulations[J]["snapshot"]
    ax.imshow(spins, cmap="hsv", vmin=0, vmax=2*np.pi, interpolation="nearest")
    ax.set_title(f"J = {J}")
    ax.set_xticks([])
    ax.set_yticks([])

# Add a colorbar to the last axis
sm = plt.cm.ScalarMappable(cmap="hsv", norm=plt.Normalize(0, 2*np.pi))
cbar = fig.colorbar(sm, ax=axes, ticks=[0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi],
                     shrink=0.6, label="Angle (rad)")
cbar.ax.set_yticklabels(["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
plt.tight_layout()
plt.show()

The energy is minimized when all spins point in the same direction. We see in the previous figures that, as the temperature is lowered, regions of neighboring spins tend to align (so we see that near spins have the same color, which corresponds to the angle).

Vortices correspond to points in the plane where colors of the full spectrum (corresponding to the angle) meet. We can use another representation of the lattice, where we attach an arrow to each spin that points in the direction corresponding to the angle $\theta_i$. We focus on a smaller portion of the lattice. If we are lucky we can see some vortices appear.

In [ ]:
# Arrow plot for the highest-J simulation (vortices visible)
J = values_of_J[-1]
spins = simulations[J]["snapshot"]
region = spins[:100, :100]
L_region = region.shape[0]

fig, ax = plt.subplots(figsize=(12, 12))
X, Y = np.meshgrid(np.arange(L_region), np.arange(L_region))
U = 0.9 * np.cos(region)
V = 0.9 * np.sin(region)
# Color arrows by angle
colors = region / (2 * np.pi)
ax.quiver(X, Y, U, V, colors, cmap="hsv", scale=L_region, width=0.003,
          headwidth=3, headlength=4, clim=(0, 1))
ax.set_title(f"J = {J} (arrow plot, 100×100 region)")
ax.set_aspect("equal")
ax.set_xlim(-1, L_region)
ax.set_ylim(-1, L_region)
plt.tight_layout()
plt.show()

At a vortex spins are far from being aligned, and thus vortices come at an energy cost. Isolated vortices are topological defects. It is impossible to apply a continuous rotation to all the spins near a vortex to align them and bring them to a ground state configuration.

To counteract this energy cost, at low temperatures vortices couple to anti-vortices, forming vortex-antivortex pairs. Arrows flowing away from an antivortex flow towards a near vortex. Such coupled vortex pairs can come together and "annihilate" each other.

The Kosterlitz–Thouless transition characterizes the emergence of these bound pairs of vortices. It was one of the earliest examples of a *topological phase transition*, described by Berezinskii, Kosterlitz, and Thouless in the 70s.